# Exercise 11.1: Z → μμ with CMS open data

From *Programming in High Energy Particle Physics*, Chapter 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch11/ex11_1.ipynb)

Exercise 11.1 Z → μμ with CMS open data. Use the public [Run2016G DoubleMuon NanoAODv9 record](https://opendata.cern.ch/record/30522) and the matching 2016 [certified-luminosity JSON record](https://opendata.cern.ch/record/14220). The selected file is 66,090,536 bytes. Read at most 200,000 events, apply the Golden JSON, require two tight opposite-charge muons, and fit their invariant-mass peak. If a download fails, an explicitly labelled synthetic fallback keeps the notebook runnable.

<details><summary>Hint</summary>

Read `Muon_pt`, `Muon_eta`, `Muon_phi`, `Muon_charge`, `Muon_tightId`, `run`, and `luminosityBlock` from `Events`. Select certified run/luminosity-section pairs before choosing the two highest-$p_T$ accepted muons. Fit a Voigtian plus background. A fit to one small uncalibrated file is an exercise, not a precision CMS result.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward vector scipy matplotlib numpy")

### Data source and fallback

The CMS file and Golden JSON are the default. A seeded synthetic dataset is permitted only if their download fails, and must be labelled.

### Step 1: Load CMS-like NanoAOD branches and apply a certified mask

In [ ]:
import json, urllib.request
import numpy as np, awkward as ak, uproot

NANOAOD_URL = "https://opendata.cern.ch/eos/opendata/cms/Run2016G/DoubleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v2/2430000/4E3A26DE-E53B-A844-8048-36376617AE8D.root"
GOLDEN_JSON_URL = "https://opendata.cern.ch/eos/opendata/cms/validated-runs/2016/Cert_271036-284044_13TeV_Legacy2016_Collisions16_JSON.txt"
BRANCHES = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge", "Muon_tightId", "run", "luminosityBlock"]
ENTRY_STOP = 200000
source = "CMS Run2016G DoubleMuon NanoAODv9"
try:
    with urllib.request.urlopen(GOLDEN_JSON_URL, timeout=30) as response:
        golden = json.load(response)
    with uproot.open(NANOAOD_URL, timeout=60) as root_file:
        events = root_file["Events"].arrays(BRANCHES, entry_stop=ENTRY_STOP, library="ak")
except (OSError, TimeoutError, urllib.error.URLError) as exc:
    print(f"CMS/Golden JSON download failed ({exc}); using synthetic fallback, not CERN data.")
    source = "synthetic fallback"
    rng = np.random.default_rng(42)
    n = 20000
    masses_seed = np.clip(91.1876 + rng.standard_cauchy(n) * 1.25 + rng.normal(0, 1.3, n), 65, 120)
    pt = masses_seed / 2
    events = ak.Array({"Muon_pt": np.stack([pt, pt], axis=1),
                       "Muon_eta": np.zeros((n, 2)),
                       "Muon_phi": np.tile([0.0, np.pi], (n, 1)),
                       "Muon_charge": np.tile([1, -1], (n, 1)),
                       "Muon_tightId": np.ones((n, 2), dtype=bool),
                       "run": np.full(n, 1),
                       "luminosityBlock": rng.integers(1, 101, n)})
    golden = {"1": [[1, 100]]}
print("Source:", source, "events read:", len(events))


### Step 2: Select tight opposite-charge muon pairs

In [ ]:
import vector
vector.register_awkward()
# TODO: apply Golden JSON run/luminosityBlock ranges.
# TODO: select the two highest-pT tight muons with pT > 25 GeV, |eta| < 2.4, opposite charge.
# TODO: form one invariant mass per accepted event and keep 70 < m_mumu < 110 GeV.

### Step 3: Fit a Voigtian and estimate uncertainty

In [ ]:
from scipy.special import voigt_profile
# TODO: fit the selected mass spectrum and estimate statistical and model uncertainties.

### Self-check
Run the cells above and inspect shapes, units and plots. Explain any mismatch with the expected physical scale.

In [ ]:
assert len(masses) > 0
assert (masses > 70).all() and (masses < 110).all()